In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("Sample - Superstore.csv", encoding="windows-1252")
df.head()

In [ ]:
df.shape


In [ ]:
df.info()

In [ ]:
df.isnull().sum(), df.duplicated().sum()

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month_Name"] = df["Order Date"].dt.strftime("%b")
df["Year_Month"] = df["Order Date"].dt.to_period("M").astype(str)
df["Profit Margin"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"] * 100, 0)
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Status"] = np.where(df["Profit"] >= 0, "Profitable", "Loss-Making")

df.head()

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
total_quantity = df["Quantity"].sum()
profit_margin = total_profit / total_sales * 100
avg_order_value = total_sales / total_orders

pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Profit Margin %", "Total Orders",
               "Total Customers", "Quantity Sold", "Average Order Value"],
    "Value": [total_sales, total_profit, profit_margin, total_orders,
              total_customers, total_quantity, avg_order_value]
})

In [ ]:
monthly = df.groupby("Year_Month", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
)

plt.figure(figsize=(12,5))
plt.plot(monthly["Year_Month"], monthly["Sales"], marker="o", label="Sales")
plt.plot(monthly["Year_Month"], monthly["Profit"], marker="o", label="Profit")
plt.xticks(rotation=45)
plt.title("Monthly Sales and Profit")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
category = df.groupby("Category", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
)
category["Profit Margin"] = category["Profit"] / category["Sales"] * 100
category.sort_values("Profit", ascending=False)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=category.sort_values("Profit", ascending=False), x="Category", y="Profit")
plt.title("Profit by Category")
plt.tight_layout()
plt.show()

In [ ]:
region = df.groupby("Region", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
)
region["Profit Margin"] = region["Profit"] / region["Sales"] * 100
region.sort_values("Profit", ascending=False)

In [ ]:
subcategory = df.groupby("Sub-Category", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
)
subcategory["Profit Margin"] = subcategory["Profit"] / subcategory["Sales"] * 100
subcategory.sort_values("Profit", ascending=False)

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=subcategory.sort_values("Profit"), y="Sub-Category", x="Profit")
plt.title("Profit by Sub-Category")
plt.tight_layout()
plt.show()

In [ ]:
discount = df.groupby("Discount", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)
discount

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Discount", y="Profit", alpha=0.4)
plt.title("Discount vs Profit")
plt.tight_layout()
plt.show()

df[["Discount", "Profit"]].corr()

In [ ]:
product = df.groupby(["Product ID", "Product Name"], as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
)

print("Top 10 by Sales")
display(product.nlargest(10, "Sales"))

print("Top 10 by Profit")
display(product.nlargest(10, "Profit"))

print("Top 10 Loss-Making Products")
display(product.nsmallest(10, "Profit"))

In [ ]:
segment = df.groupby("Segment", as_index=False).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
)
segment["Profit Margin"] = segment["Profit"] / segment["Sales"] * 100
segment.sort_values("Profit", ascending=False)

In [ ]:
best_category = category.loc[category["Profit"].idxmax()]
worst_category = category.loc[category["Profit"].idxmin()]
best_region = region.loc[region["Profit"].idxmax()]
worst_region = region.loc[region["Profit"].idxmin()]
best_subcategory = subcategory.loc[subcategory["Profit"].idxmax()]
worst_subcategory = subcategory.loc[subcategory["Profit"].idxmin()]
loss_products = (product["Profit"] < 0).sum()
discount_corr = df[["Discount", "Profit"]].corr().iloc[0,1]

print(f"1. {best_category['Category']} has the highest category profit: {best_category['Profit']:,.2f}.")
print(f"2. {worst_category['Category']} has the lowest category profit: {worst_category['Profit']:,.2f}.")
print(f"3. {best_region['Region']} is the most profitable region: {best_region['Profit']:,.2f}.")
print(f"4. {best_subcategory['Sub-Category']} is the most profitable sub-category: {best_subcategory['Profit']:,.2f}.")
print(f"5. {loss_products} products have negative total profit.")
print(f"Discount-profit correlation: {discount_corr:.2f}")

In [ ]:
recommendations = [
    "Review high-discount transactions and reduce discounts where profitability is weak.",
    "Investigate loss-making products and sub-categories for pricing and cost issues.",
    "Focus growth efforts on high-profit categories, products and regions."
]

for i, r in enumerate(recommendations, 1):
    print(f"{i}. {r}")

In [ ]:
output = Path("outputs")
output.mkdir(exist_ok=True)

df.to_csv(output / "Cleaned_Superstore.csv", index=False)
monthly.to_csv(output / "Monthly_Analysis.csv", index=False)
category.to_csv(output / "Category_Analysis.csv", index=False)
region.to_csv(output / "Region_Analysis.csv", index=False)
subcategory.to_csv(output / "Subcategory_Analysis.csv", index=False)
product.to_csv(output / "Product_Analysis.csv", index=False)
segment.to_csv(output / "Segment_Analysis.csv", index=False)

print("Files saved in outputs/")